In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 3
NUM_CLASSES = 5 # Bike, Car, FighterJet, Helicopter, Tank (5 классов)
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 50 # Максимальное число эпох (ранняя остановка сработает раньше)
SEED = 42
DATA_DIR = 'dataset' # Убедитесь, что папка 'dataset' находится в текущей директории

In [6]:
def create_custom_cnn(input_shape, num_classes):
    """
    Создает кастомную CNN архитектуру (4 сверточных блока).
    """
    model = keras.Sequential([
        # Входной слой (224, 224, 3)
        layers.Input(shape=input_shape),
        
        # --- Блок 1: 32 фильтра ---
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(), 
        layers.MaxPooling2D((2, 2)), # Размер уменьшился
        
        # --- Блок 2: 64 фильтра ---
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)), # Размер уменьшился
        
        # --- Блок 3: 128 фильтров ---
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)), # Размер уменьшился

        # --- Блок 4: 128 фильтров ---
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)), # Размер уменьшился
        
        # --- Классификатор ---
        layers.Flatten(),                  
        layers.Dense(128, activation='relu'), 
        layers.Dropout(0.5),               # Защита от переобучения
        layers.Dense(num_classes, activation='softmax') 
    ])
    
    return model

In [7]:
print("Загрузка обучающей выборки...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'),
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH), # Приводим к единому размеру
    batch_size=BATCH_SIZE
)

print("Загрузка валидационной выборки...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'validation'),
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

print("Загрузка тестовой выборки...")
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'test'),
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

Загрузка обучающей выборки...
Found 3000 files belonging to 5 classes.
Загрузка валидационной выборки...
Found 505 files belonging to 5 classes.
Загрузка тестовой выборки...
Found 505 files belonging to 5 classes.


In [8]:
# Создаем словарь для сопоставления меток и названий классов (для отчета)
class_names = train_ds.class_names
print(f"Классы: {class_names}")

Классы: ['Bike', 'Car', 'FighterJet', 'Helicopter', 'Tank']


In [9]:
# Создаем слой нормализации, который будет преобразовывать пиксели [0-255] в [0-1]
# Это должно быть первым слоем в модели или перед подачей данных.
normalization_layer = layers.Rescaling(1./255)

# Применяем нормализацию ко всем датасетам
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

# Оптимизация производительности: кэширование и предварительная загрузка данных
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [10]:
# Создание модели
input_shape = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)
model = create_custom_cnn(input_shape, NUM_CLASSES)
print("\n--- Сводка по архитектуре ---")
model.summary()

# Компиляция модели
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE) 
# Используем Sparse Categorical Crossentropy, т.к. метки - целые числа
loss = 'sparse_categorical_crossentropy' 
metrics = ['accuracy',  
          keras.metrics.Precision(name='precision'), 
          keras.metrics.Recall(name='recall'), 
          keras.metrics.AUC(name='auc')] # AUC - отличная метрика для проверки качества

model.compile(optimizer=optimizer, 
              loss=loss, 
              metrics=metrics)


--- Сводка по архитектуре ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,277 (13.18 MB)

 Trainable params: 3,453,573 (13.17 MB)

 Non-trainable params: 704 (2.75 KB)


--- Начинаем обучение модели ---


In [12]:
model.load_weights('best_model.h5') 

print("\n--- Оценка модели на тестовой выборке ---")
loss, acc, precision, recall, auc = model.evaluate(test_ds)

print(f"\nИтоговые метрики на TEST (для отчета):")
print(f"Loss (Потери): {loss:.4f}")
print(f"Accuracy (Точность): {acc:.4f}")
print(f"Precision (Точность предсказаний): {precision:.4f}")
print(f"Recall (Полнота): {recall:.4f}")
print(f"AUC (Площадь под кривой): {auc:.4f}")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'best_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)